In [5]:
import wandb
wandb.init(mode="disabled", project="qa_squad_demo")
#!pip install transformers datasets torch evaluate rouge_score
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# 1. Load a small subset of data
dataset = load_dataset("cnn_dailymail", '3.0.0')
train_data = dataset["train"].select(range(100))
val_data = dataset["validation"].select(range(10))

# 2. Initialize model and tokenizer
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 3. Preprocess function
def preprocess(examples):
    inputs = ["summarize: " + text for text in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(examples["highlights"], max_length=64, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 4. Preprocess datasets
train_dataset = train_data.map(preprocess, batched=True)
val_dataset = val_data.map(preprocess, batched=True)

# 5. Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    learning_rate=2e-5,
    logging_dir="./logs"
)

# 6. Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 7. Create trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

# 8. Train
trainer.train()

# 9. Test the model
test_article = """summarize: Scientists have discovered a new species of dinosaur in Argentina.
The fossils, found in Patagonia, suggest this dinosaur was one of the largest ever discovered.
Initial estimates indicate it was over 100 feet long."""

# Move input to same device as model
inputs = tokenizer(test_article, return_tensors="pt").to(device)
outputs = model.generate(inputs.input_ids, max_length=50)
print("\nTest Summary:", tokenizer.decode(outputs[0], skip_special_tokens=True))

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss



Test Summary: fossils found in argentina suggest this dinosaur was one of the largest ever discovered. initial estimates indicate it was over 100 feet long.
